<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [3]:
from pathlib import Path
print(Path.cwd())

/home/jose/projects/ibmdecert_capstoneprj/module06


In [1]:
# Start session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Ecommerce-Search-Analysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/20 15:23:13 WARN Utils: Your hostname, user, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/20 15:23:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 15:23:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# Load the csv into a spark dataframe
df = spark.read.csv("searchterms.csv", header=True, inferSchema=True)

In [6]:
# Print the number of rows and columns
rows = df.count()
cols = len(df.columns)

print(f"Filas: {rows}, Columnas: {cols}")

Filas: 10000, Columnas: 4


In [7]:
# Print the top 5 rows
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows


In [8]:
# Find out the datatype of the column searchterm?
df.select("searchterm").printSchema()

root
 |-- searchterm: string (nullable = true)



In [9]:
# How many times was the term `gaming laptop` searched?
from pyspark.sql.functions import col

gaming_laptop_count = df.filter(col("searchterm") == "gaming laptop").count()
print(f"El término 'gaming laptop' fue buscado {gaming_laptop_count} veces.")

El término 'gaming laptop' fue buscado 499 veces.


In [10]:
# Print the top 5 most frequently used search terms?
from pyspark.sql.functions import desc

df.groupBy("searchterm") \
  .count() \
  .orderBy(desc("count")) \
  .show(5)

+-------------+-----+
|   searchterm|count|
+-------------+-----+
|mobile 6 inch| 2312|
|    mobile 5g| 2301|
|mobile latest| 1327|
|       laptop|  935|
|  tablet wifi|  896|
+-------------+-----+
only showing top 5 rows


In [14]:
# Load the sales forecast model.
from pyspark.ml.regression import LinearRegressionModel

# Ajusta "model" al nombre exacto de la carpeta que se genere tras descomprimir
model = LinearRegressionModel.load("sales_prediction.model")

In [15]:
# Using the sales forecast model, predict the sales for the year of 2023.
from pyspark.ml.feature import VectorAssembler

# 1. Creamos un DataFrame con el año que queremos predecir (2023)
# Nota: Revisa si el modelo original usaba el nombre de columna 'year' u otro similar
data_2023 = spark.createDataFrame([(2023,)], ["year"])

# 2. Vectorizamos la columna 'year' para que el modelo la entienda
assembler = VectorAssembler(inputCols=["year"], outputCol="features")
features_2023 = assembler.transform(data_2023)

# 3. Realizamos la predicción
predictions = model.transform(features_2023)

# 4. Mostramos el resultado (la columna 'prediction' contendrá el valor esperado)
predictions.select("year", "prediction").show()

+----+------------------+
|year|        prediction|
+----+------------------+
|2023|175.16564294006457|
+----+------------------+



26/06/20 17:27:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
